# Stateless operators

Stateless operators are "stateless" in that they do not accumulate state. Simply put, a topology only consisting of stateless operators can swallow arbitrary many messages from the sources without ever running out of memory.


## Overview

[Prepation](#prep)

* [map()](#map-operator)
* [peek()](#peek-operator)
* [flatmap()](#flatmap-operator)
* [filter()](#filter-operator)
* [merge()](#merge-operator)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [1]:
!pip install -r ../requirements.txt

import sys
sys.path.insert(1, "../")
sys.path.insert(1, "../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()

click_source_str = "clicks"
customer_source_str = "customers"



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Please also note that when we re-use the same example over and over again to illustrate how the operators work, we always mark the important new parts as follows:
```python
    # <------------------------------>
    ...important new parts...
    # <------------------------------>
```

---
<a id="map-operator"></a>
## map()

Classic `map()` operator, like e.g. in Kafka Streams.

```
map(map_fun, **kwargs)
```
* `map_fun: r -> r` the map function; gets an input record, does some processing and returns an output record.

Here is an example.

In [4]:
built_tn = Tn.build(
    Tn.source(click_source_str)
    #
    # <------------------------------>
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
    # <------------------------------>
)

input_m_list = click_generator.generate(5)
print("Input:")
for m in input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Input:
{'key': None, 'value': {'customer_id': 68, 'view_time': 80, 'ts': 1787518302735}}
{'key': None, 'value': {'customer_id': 75, 'view_time': 107, 'ts': 1787518302835}}
{'key': None, 'value': {'customer_id': 0, 'view_time': 111, 'ts': 1787518302935}}
{'key': None, 'value': {'customer_id': 69, 'view_time': 55, 'ts': 1787518303035}}
{'key': None, 'value': {'customer_id': 2, 'view_time': 100, 'ts': 1787518303135}}

Output:
{'customer_id': 68, 'view_time': 80}
{'customer_id': 75, 'view_time': 107}
{'customer_id': 0, 'view_time': 111}
{'customer_id': 69, 'view_time': 55}
{'customer_id': 2, 'view_time': 100}


Simple. We pushed a few messages to the topology and `map()` selected two fields from them:

```mermaid
graph TD
ba9b2d80-3f93-4b57-9eb6-d02ff933493d[source_clicks] --> 1ab0b693-3e7d-486c-a2d7-6596a8671cc9[map_op]
```

---
<a id="peek-operator"></a>
## peek()

This operator is mainly for debugging purposes. Under the covers, it's a `print` + an optional `None`-returning function that can be used to trigger side effects.

```
peek(prefix_str=None, peek_fun=None, **kwargs)
```
* `prefix_str` prefix of the printed output (if `peek_fun` is `None`)
* `peek_fun: r -> None` the `None`-returning peek function getting the input record

Here are a few examples. The first sets none of the parameters.

In [ ]:
built_tn = Tn.build(
    Tn.source(click_source_str)
    #
    # <------------------------------>
    .peek()
    # <------------------------------>
)

m_list = click_generator.generate(5)

_ = built_tn.process({click_source_str: m_list})


You can see that `peek()` just printed out each of the records coming in.

Here is the topology graph - you can see that the `peek()` is actually nothing else but a `map()` under the covers:

```mermaid
graph TD
ba9b2d80-3f93-4b57-9eb6-d02ff933493d[source_clicks] --> 1ab0b693-3e7d-486c-a2d7-6596a8671cc9[map_op]
```

Next, we use `peek` with `prefix_str` set:

In [ ]:
built_tn = Tn.build(
    Tn.source(click_source_str)
    #
    # <------------------------------>
    .peek("peek")
    # <------------------------------>
)

m_list = click_generator.generate(5)

_ = built_tn.process({click_source_str: m_list})


...and you can see that `peek()` now adds the prefix `peek: ` to the input records printed out.

Last example: We use the `peek_fun`:

In [ ]:
built_tn = Tn.build(
    Tn.source(click_source_str)
    #
    # <------------------------------>
    .peek(peek_fun=lambda r: print(f"{r}\n"))
    # <------------------------------>
)

m_list = click_generator.generate(5)

_ = built_tn.process({click_source_str: m_list})



What we did in the `peek_fun` is to add a newline after each printed out input record.

---
<a id="flatmap-operator"></a>
## flatmap()

This is the relational version of the classic `flatmap` operator also known from e.g. Kafka Streams. It is relational in the sense that its return value (an iterable) is interpreted as a set of outputs instead of as a list.

```
flatmap(flatmap_fun, **kwargs)
```
* `flatmap_fun: r -> iterable(r)` the flatmap function; gets an input record and returns an iterable (list or set etc.) of output records.

Examples.

In [8]:
built_tn = Tn.build(
    Tn.source(customer_source_str)
    #
    # <------------------------------>
    .flatmap(lambda r: {name_part_str for name_part_str in r["value"]["name"].split(" ")})
    # <------------------------------>
)

input_m_list = customer_generator.generate(5)
print("Input:")
for m in input_m_list:
    print(m)

output_m_list = built_tn.process({customer_source_str: input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Input:
{'key': '36', 'value': {'id': 36, 'name': 'Amy Smith'}}
{'key': '1', 'value': {'id': 1, 'name': 'Austin Buchanan'}}
{'key': '97', 'value': {'id': 97, 'name': 'Abigail Hernandez'}}
{'key': '90', 'value': {'id': 90, 'name': 'Christopher Chen'}}
{'key': '10', 'value': {'id': 10, 'name': 'Jeffrey Singleton MD'}}

Output:
Amy
Smith
Buchanan
Austin
Abigail
Hernandez
Christopher
Chen
Jeffrey
Singleton
MD


In this example, from each customer input record, we take the `name` field of its `value`, split it and return the set of the parts of the name. So e.g., `"Alexis Jones"` becomes `{"Alexis", "Jones"}`.

Here is the Mermaid representation of this topology:
```mermaid
graph TD
f2306785-a69c-4dfc-8ad3-4e8fd593dd4a[source_customers] --> 9ad0babc-b672-4d6b-a968-f33ab51cb164[flatmap_op]
```

A classical flatmap returns a list. In Kafi Streams, being based on a relational engine that is pydbsp, it is an iterable (list or set etc.), but the key point is that this iterable is always interpreted as a *set*: The order of the resulting iterable is not preserved + duplicates are automatically removed.

To see this clearly, look at the following example where we stitch together an input message where the first name is the same as the last name, and we add a third name component as well:

In [5]:
built_tn = Tn.build(
    Tn.source(customer_source_str)
    #
    # <------------------------------>
    .flatmap(lambda r: {name_part_str for name_part_str in r["value"]["name"].split(" ")})
    # <------------------------------>
)

input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Frank Frank Anna'}}]
print("Input:")
print(input_m_list)

output_m_list = built_tn.process({customer_source_str: input_m_list})
print("Output:")
print(output_m_list)


Input:
[{'key': '42', 'value': {'id': 42, 'name': 'Frank Frank Anna'}}]
Output:
['Anna', 'Frank']


As you can observe:
* the order of the input string (`Frank` came in before `Anna`) is not preserved,
* and the duplicate occurrence of `Frank` is automatically de-duplicated

Welcome to set theory ;-)

The "philosophy" of Kafi Streams is relational/set-based for a reason. This is not a defect but intentional. The entire mathematical foundation of the underlying pydbsp engine rests on it.

If you miss your "classical" list-returning flatmap, don't despair - you can still recover it e.g. as below:

In [6]:
built_tn = Tn.build(
    Tn.source(customer_source_str)
    #
    # <------------------------------>
    .flatmap(lambda r: {(i, name_part_str) for i, name_part_str in enumerate(r["value"]["name"].split(" "))})
    # <------------------------------>
)

input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Frank Frank Anna'}}]
print("Input:")
print(input_m_list)

output_m_list = built_tn.process({customer_source_str: input_m_list})
print("Output:")
print(output_m_list)


Input:
[{'key': '42', 'value': {'id': 42, 'name': 'Frank Frank Anna'}}]
Output:
[[1, 'Frank'], [2, 'Anna'], [0, 'Frank']]


---
<a id="filter-operator"></a>
## filter()

On to the next classical stateless operator: `filter`.

```
filter(filter_fun, **kwargs)
```
* `filter_fun: r -> bool` the filter function; gets an input record and returns `True` if the record shall be kept or `False` if it shall be discarded.

Here is an example.


In [11]:
built_tn = Tn.build(
    Tn.source(click_source_str)
    #
    # <------------------------------>
    .filter(lambda r: r["value"]["view_time"] > 60)
    # <------------------------------>
)

input_m_list = click_generator.generate(5)
print("Input:")
for m in input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Input:
{'key': None, 'value': {'customer_id': 93, 'view_time': 39, 'ts': 1787518303235}}
{'key': None, 'value': {'customer_id': 19, 'view_time': 50, 'ts': 1787518303335}}
{'key': None, 'value': {'customer_id': 31, 'view_time': 114, 'ts': 1787518303435}}
{'key': None, 'value': {'customer_id': 98, 'view_time': 56, 'ts': 1787518303535}}
{'key': None, 'value': {'customer_id': 81, 'view_time': 55, 'ts': 1787518303635}}

Output:
{'key': None, 'value': {'customer_id': 31, 'view_time': 114, 'ts': 1787518303435}}


In the example, we just keep those records where `view_time` is greater than `60`.

Here is the Mermaid representation of the topology:
```mermaid
graph TD
0dab1488-2505-4a9d-a9f6-7294dc3e42c8[source_clicks] --> 54ad4109-956d-404b-9520-04fd10d1bf20[filter_op]
```

---
<a id="merge-operator"></a>
## merge()

Similar to Kafka Streams, the purpose of `merge` is to combine two branches of your topology.

```
merge(other_tn, **kwargs):
```
* `other_tn` the other topology node that the current shall be merged with

This screams for an example.


In [2]:
click_tn = (
    Tn.source(click_source_str)
    #
    # <------------------------------>
    .map(lambda r: {"id": r["value"]["customer_id"]})
    # <------------------------------>
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    # <------------------------------>
    .map(lambda r: {"id": r["value"]["id"]})
    # <------------------------------>
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .merge(customer_tn)
    # <------------------------------>
)

click_input_m_list = click_generator.generate(5)
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = customer_generator.generate(5)
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Input (clicks):
{'key': None, 'value': {'customer_id': 57, 'view_time': 100, 'ts': 1787518302235}}
{'key': None, 'value': {'customer_id': 95, 'view_time': 94, 'ts': 1787518302335}}
{'key': None, 'value': {'customer_id': 79, 'view_time': 93, 'ts': 1787518302435}}
{'key': None, 'value': {'customer_id': 70, 'view_time': 48, 'ts': 1787518302535}}
{'key': None, 'value': {'customer_id': 65, 'view_time': 12, 'ts': 1787518302635}}

Input (customers):
{'key': '30', 'value': {'id': 30, 'name': 'Dennis Russell'}}
{'key': '70', 'value': {'id': 70, 'name': 'Alexander Gould'}}
{'key': '99', 'value': {'id': 99, 'name': 'Brandon Lopez'}}
{'key': '81', 'value': {'id': 81, 'name': 'Linda Kennedy'}}
{'key': '59', 'value': {'id': 59, 'name': 'Isaac Estrada'}}

Output:
{'id': 57}
{'id': 95}
{'id': 79}
{'id': 70}
{'id': 70}
{'id': 65}
{'id': 30}
{'id': 99}
{'id': 81}
{'id': 59}


This is what happens:
* We create two sub topologies - one for the clicks (`click_tn`) and one for the customers (`customer_tn`).
* In each sub topology, we select just the customer ID from the input records (`customer_id` for the clicks, `id` for the customers).
* We then use the `merge()` operator to merge the outputs of the two sub topologies together.

Here is the Mermaid representation:

```mermaid
graph TD
da7fe5fc-a151-4e68-8eaa-968befa3493d[source_clicks] --> 95269802-5d4f-4da4-ad3b-7d8d1b01180f[map_op]
3b8e544c-2496-41e0-965c-e08ecb3e0851[source_customers] --> 3c9d2262-43b3-4b45-b669-38930e52f349[map_op]
95269802-5d4f-4da4-ad3b-7d8d1b01180f[map_op] --> c9967e4e-f8e6-40e7-835d-a588f3df2cf1[merge_op]
3c9d2262-43b3-4b45-b669-38930e52f349[map_op] --> c9967e4e-f8e6-40e7-835d-a588f3df2cf1[merge_op]
```

`merge` is a stateless operation. It is neither a union or a join. Under the covers, it just adds up the weights of the ZSets of the input records:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"id": r["value"]["customer_id"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    .map(lambda r: {"id": r["value"]["id"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .merge(customer_tn)
    # <------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 76, 'ts': 1786618958910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


In the latest example, we just sent one message to each source, both having the same customer ID (`42`). The result are *two* records, not one (under the covers, it's actually one record with weight `2`).